# 🤖 Clase 2: Agentes y Herramientas (Tools)

## Bienvenido a la Semana 2, Clase 2

En esta clase aprenderás:
- ✅ ¿Qué son los agentes de IA?
- ✅ ReAct: Reasoning + Acting
- ✅ Tools: Herramientas para agentes
- ✅ Function calling
- ✅ Crear un agente con búsqueda web (Tavily)
- ✅ Chain of Thought

---

In [ ]:
# Instalación
!pip install langchain langchain-openai tavily-python langsmith -q

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain.tools import Tool
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

load_dotenv()

llm = ChatOpenAI(model="gpt-4", temperature=0)
print("✅ Configuración completada")

## 🤖 Parte 1: ¿Qué es un Agente?

### Diferencia: Chain vs Agente

**Chain (Cadena)**:
```
Input → Paso 1 → Paso 2 → Paso 3 → Output
(Siempre el mismo camino)
```

**Agente**:
```
Input → Pensar → ¿Qué hacer?
           ↓
      Usar Tool A → Pensar → ¿Listo?
           ↓                    ↓
      Usar Tool B → Pensar → Output
(El agente decide qué hacer)
```

### Características de un Agente

1. **Autonomía**: Decide qué herramientas usar
2. **Razonamiento**: Piensa paso a paso
3. **Iteración**: Puede usar múltiples herramientas
4. **Adaptabilidad**: Se ajusta según los resultados

## 🛠️ Parte 2: Tools (Herramientas)

Las **tools** son funciones que el agente puede usar.

In [ ]:
# Tool simple: Calculadora
def calculadora(expresion: str) -> str:
    """Evalúa una expresión matemática."""
    try:
        resultado = eval(expresion)
        return f"El resultado es: {resultado}"
    except Exception as e:
        return f"Error: {e}"

# Crear Tool
tool_calculadora = Tool(
    name="Calculadora",
    description="Útil para realizar cálculos matemáticos. Input debe ser una expresión matemática válida como '2+2' o '10*5'.",
    func=calculadora
)

# Probar
print(tool_calculadora.run("25 * 4 + 10"))

In [ ]:
# Más tools de ejemplo
from datetime import datetime
import random

def obtener_fecha() -> str:
    """Obtiene la fecha y hora actual."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def lanzar_dado() -> str:
    """Lanza un dado de 6 caras."""
    resultado = random.randint(1, 6)
    return f"🎲 Resultado del dado: {resultado}"

def contar_palabras(texto: str) -> str:
    """Cuenta las palabras en un texto."""
    palabras = len(texto.split())
    caracteres = len(texto)
    return f"Palabras: {palabras}, Caracteres: {caracteres}"

# Crear tools
tools = [
    Tool(
        name="Calculadora",
        description="Útil para cálculos matemáticos. Input: expresión matemática.",
        func=calculadora
    ),
    Tool(
        name="ObtenerFecha",
        description="Obtiene la fecha y hora actual. No requiere input.",
        func=lambda x: obtener_fecha()
    ),
    Tool(
        name="LanzarDado",
        description="Lanza un dado de 6 caras. No requiere input.",
        func=lambda x: lanzar_dado()
    ),
    Tool(
        name="ContarPalabras",
        description="Cuenta palabras y caracteres en un texto. Input: texto a analizar.",
        func=contar_palabras
    )
]

print(f"✅ {len(tools)} herramientas creadas")

## 🧠 Parte 3: Crear un Agente

### Prompt del Agente

In [ ]:
# Crear prompt para el agente
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente útil que puede usar herramientas para responder preguntas.
    
Tienes acceso a las siguientes herramientas:
{tools}

Usa las herramientas cuando sea necesario. Piensa paso a paso.
Si no necesitas una herramienta, responde directamente."""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

print("✅ Prompt del agente creado")

In [ ]:
# Crear agente
agent = create_openai_functions_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

# Crear executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  # Muestra el razonamiento
    max_iterations=5,  # Máximo de iteraciones
    handle_parsing_errors=True
)

print("✅ Agente creado y listo para usar")

In [ ]:
# Probar el agente
preguntas = [
    "¿Cuánto es 15 * 8 + 23?",
    "¿Qué fecha es hoy?",
    "Lanza un dado y dime el resultado",
    "Cuenta las palabras en esta frase: La inteligencia artificial es fascinante"
]

for pregunta in preguntas:
    print(f"\n{'='*80}")
    print(f"❓ Pregunta: {pregunta}")
    print("="*80)
    
    resultado = agent_executor.invoke({"input": pregunta})
    print(f"\n🤖 Respuesta: {resultado['output']}")

## 🔍 Parte 4: ReAct - Reasoning + Acting

**ReAct** es un patrón donde el agente:

1. **Thought (Pensamiento)**: Razona sobre qué hacer
2. **Action (Acción)**: Ejecuta una herramienta
3. **Observation (Observación)**: Ve el resultado
4. Repite hasta tener la respuesta

### Ejemplo de ReAct

```
Pregunta: "¿Cuánto es 25 * 4 y qué día es hoy?"

Thought: Necesito hacer dos cosas: un cálculo y obtener la fecha
Action: Usar Calculadora con "25 * 4"
Observation: El resultado es 100

Thought: Ahora necesito la fecha
Action: Usar ObtenerFecha
Observation: 2024-02-03

Thought: Tengo toda la información
Answer: 25 * 4 = 100, y hoy es 2024-02-03
```

In [ ]:
# Pregunta compleja que requiere múltiples herramientas
pregunta_compleja = """Haz lo siguiente:
1. Calcula 50 * 3
2. Dime qué fecha es hoy
3. Lanza un dado
4. Dame un resumen de todo"""

print("🧠 Ejecutando pregunta compleja...\n")
resultado = agent_executor.invoke({"input": pregunta_compleja})
print(f"\n✅ Respuesta final:\n{resultado['output']}")

## 🌐 Parte 5: Agente con Búsqueda Web (Tavily)

**Tavily** es una API de búsqueda web optimizada para LLMs.

In [ ]:
# Configurar Tavily
from langchain_community.tools.tavily_search import TavilySearchResults

# Crear tool de búsqueda
search_tool = TavilySearchResults(
    max_results=3,
    api_key=os.getenv("TAVILY_API_KEY")
)

# Agregar a las tools
tools_con_busqueda = tools + [search_tool]

print(f"✅ Tool de búsqueda web agregada")
print(f"Total de herramientas: {len(tools_con_busqueda)}")

In [ ]:
# Crear agente con búsqueda web
agent_web = create_openai_functions_agent(
    llm=llm,
    tools=tools_con_busqueda,
    prompt=prompt
)

agent_web_executor = AgentExecutor(
    agent=agent_web,
    tools=tools_con_busqueda,
    verbose=True,
    max_iterations=5
)

print("✅ Agente con búsqueda web creado")

In [ ]:
# Probar búsqueda web
preguntas_web = [
    "¿Cuál es la última versión de Python?",
    "¿Qué pasó hoy en el mundo de la tecnología?",
    "Busca información sobre GPT-4 Turbo"
]

for pregunta in preguntas_web:
    print(f"\n{'='*80}")
    print(f"🔍 Pregunta: {pregunta}")
    print("="*80)
    
    resultado = agent_web_executor.invoke({"input": pregunta})
    print(f"\n🤖 Respuesta:\n{resultado['output']}")

## 🎯 Parte 6: Tools Personalizadas Avanzadas

In [ ]:
# Tool que accede a una "base de datos" ficticia
productos_db = {
    "laptop": {"nombre": "Laptop Pro", "precio": 1200, "stock": 5},
    "mouse": {"nombre": "Mouse Inalámbrico", "precio": 25, "stock": 50},
    "teclado": {"nombre": "Teclado Mecánico", "precio": 80, "stock": 15}
}

def consultar_producto(producto: str) -> str:
    """Consulta información de un producto en la base de datos."""
    producto = producto.lower().strip()
    
    if producto in productos_db:
        info = productos_db[producto]
        return f"""Producto: {info['nombre']}
Precio: ${info['precio']}
Stock disponible: {info['stock']} unidades"""
    else:
        return f"Producto '{producto}' no encontrado. Productos disponibles: {', '.join(productos_db.keys())}"

def calcular_descuento(precio: str, descuento: str) -> str:
    """Calcula el precio con descuento. Input: 'precio,descuento' (ej: '100,20' para $100 con 20% descuento)."""
    try:
        precio_num, desc_num = map(float, precio.split(','))
        precio_final = precio_num * (1 - desc_num/100)
        ahorro = precio_num - precio_final
        return f"Precio original: ${precio_num}\nDescuento: {desc_num}%\nPrecio final: ${precio_final:.2f}\nAhorro: ${ahorro:.2f}"
    except:
        return "Error: Formato debe ser 'precio,descuento' (ej: '100,20')"

# Crear tools
tools_tienda = [
    Tool(
        name="ConsultarProducto",
        description="Consulta información de un producto (laptop, mouse, teclado). Input: nombre del producto.",
        func=consultar_producto
    ),
    Tool(
        name="CalcularDescuento",
        description="Calcula precio con descuento. Input: 'precio,descuento' (ej: '100,20' para $100 con 20% descuento).",
        func=calcular_descuento
    ),
    Tool(
        name="Calculadora",
        description="Cálculos matemáticos. Input: expresión matemática.",
        func=calculadora
    )
]

print("✅ Tools de tienda creadas")

In [ ]:
# Crear agente de tienda
prompt_tienda = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente de ventas de una tienda de tecnología.
Ayudas a los clientes a consultar productos y calcular precios.

Herramientas disponibles:
{tools}

Sé amable y profesional."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent_tienda = create_openai_functions_agent(
    llm=llm,
    tools=tools_tienda,
    prompt=prompt_tienda
)

agent_tienda_executor = AgentExecutor(
    agent=agent_tienda,
    tools=tools_tienda,
    verbose=True
)

print("✅ Agente de tienda creado")

In [ ]:
# Probar agente de tienda
consultas_tienda = [
    "¿Cuánto cuesta la laptop?",
    "Si compro 3 mouse, ¿cuánto pagaría en total?",
    "¿Cuál sería el precio del teclado con 15% de descuento?"
]

for consulta in consultas_tienda:
    print(f"\n{'='*80}")
    print(f"👤 Cliente: {consulta}")
    print("="*80)
    
    resultado = agent_tienda_executor.invoke({"input": consulta})
    print(f"\n🛍️ Asistente: {resultado['output']}")

## 💡 Ejercicios Prácticos

In [ ]:
# Ejercicio 1: Crea tu propia tool
# Ejemplo: Una tool que convierta temperaturas

def convertir_temperatura(input_str: str) -> str:
    """
    Convierte temperatura entre Celsius y Fahrenheit.
    Input: 'valor,unidad' (ej: '25,C' o '77,F')
    """
    try:
        valor, unidad = input_str.split(',')
        valor = float(valor)
        unidad = unidad.strip().upper()
        
        if unidad == 'C':
            fahrenheit = (valor * 9/5) + 32
            return f"{valor}°C = {fahrenheit:.1f}°F"
        elif unidad == 'F':
            celsius = (valor - 32) * 5/9
            return f"{valor}°F = {celsius:.1f}°C"
        else:
            return "Error: Unidad debe ser 'C' o 'F'"
    except:
        return "Error: Formato debe ser 'valor,unidad' (ej: '25,C')"

# 👉 Crea un agente que use esta tool
tool_temperatura = Tool(
    name="ConvertirTemperatura",
    description="Convierte temperatura entre Celsius y Fahrenheit. Input: 'valor,unidad' (ej: '25,C' o '77,F').",
    func=convertir_temperatura
)

# Pruébalo
print(tool_temperatura.run("25,C"))
print(tool_temperatura.run("77,F"))

In [ ]:
# Ejercicio 2: Agente con memoria (conversación)
from langchain.memory import ConversationBufferMemory
from langchain.agents import initialize_agent, AgentType

# 👉 Crea un agente que recuerde la conversación
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Agente conversacional
agent_conversacional = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    memory=memory,
    verbose=True
)

# Conversación de ejemplo
print("Conversación 1:")
print(agent_conversacional.run("Mi nombre es Juan"))

print("\nConversación 2:")
print(agent_conversacional.run("¿Cuál es mi nombre?"))

## 🎓 Resumen

### Conceptos Clave

1. **Agentes**: Sistemas que deciden qué hacer
2. **Tools**: Funciones que los agentes pueden usar
3. **ReAct**: Patrón de razonamiento + acción
4. **Function Calling**: LLM invoca funciones
5. **Tavily**: Búsqueda web para agentes
6. **AgentExecutor**: Ejecuta el agente con límites

### Mejores Prácticas

- ✅ Descripciones claras de tools
- ✅ Manejo de errores en tools
- ✅ Límite de iteraciones
- ✅ Verbose=True para debugging
- ✅ Temperatura baja (0-0.3) para agentes

### Próxima Semana

En **Semana 3** aprenderemos:
- 📊 LangGraph en profundidad
- 🔄 State management
- 🤖 Sistemas multi-agente
- 📈 LangSmith para monitoreo

---

**¡Excelente trabajo! 🚀**